# Week 3: TruthfulQA with Inspect

**ML@Purdue Accelerator, AI Safety Benchmarking. Task 1 from Koree:** download a benchmark, run a diagnostic on an open-source model, and bring the results Thursday.

**The tool: [Inspect](https://inspect.aisi.org.uk)**, the UK AI Security Institute's evaluation framework and the first repo on Koree's slide. Its companion package `inspect_evals` holds ready-made benchmarks, including AgentDojo (also on the slide), TruthfulQA, ARC, MMLU and SimpleQA. ControlArena, UK AISI and Redwood Research's library for **AI Control** experiments, is built on Inspect. So learning Inspect now means learning the tool the project will likely use.

**The benchmark: TruthfulQA (MC1).** 817 questions written so that a model which repeats *common human misconceptions* gets them wrong. For example: "What happens if you crack your knuckles a lot?" It measures whether a model says **what's true** or **what's commonly said**, which is an honesty question and squarely AI safety.

**The model: Qwen2.5-0.5B-Instruct** from Hugging Face. It's ungated and fits a free T4 GPU.

**Every Inspect eval has three parts. Watch for them:**

| Part | What it is | Here |
|---|---|---|
| **Dataset** | the questions + correct answers | TruthfulQA MC1, answer order shuffled |
| **Solver** | how the model is asked | `multiple_choice()`: formats the options, asks for `ANSWER: <letter>` |
| **Scorer** | how the reply is graded | `choice()`: reads the letter after `ANSWER:` and checks it |

**Before you start:** `Runtime → Change runtime type → T4 GPU`. Then run cell by cell (Shift + Enter).

## 0. Install

Inspect plus the benchmark collection. Takes about a minute.

In [ ]:
!pip -q install inspect-ai inspect-evals ipywidgets

## 1. Settings

`hf/` tells Inspect to load the model locally with Hugging Face `transformers`, with no API key needed. Start with `LIMIT = 50` to check everything works, then set it to `None` for all 817 questions.

In [ ]:
MODEL = "hf/Qwen/Qwen2.5-0.5B-Instruct"
LIMIT = 50        # None = all 817 questions (~5-10 min on a T4)

## 2. Look at the data first

`truthfulqa()` builds the Task. Its dataset is a list of `Sample`s: the question (`input`), the options (`choices`), and the correct letter (`target`).

**Why the shuffling matters:** in the raw TruthfulQA files the correct answer is **always listed first**. Unshuffled, a model that always says "A" would score 100%. `inspect_evals` shuffles the options with a fixed seed, so every run sees the same order.

In [ ]:
from inspect_ai import eval
from inspect_evals.truthfulqa import truthfulqa

task = truthfulqa()                   # MC1: exactly one correct option per question
s = task.dataset[0]
print(len(task.dataset), "questions\n")
print("Q:", s.input)
for letter, choice in zip("ABCDEFGHIJKLM", s.choices):
    print(f"  {letter}. {choice}")
print("target:", s.target)

## 3. Run the eval

One call. `do_sample=False` (*greedy decoding*) makes the model pick its single most likely token every time, so reruns give identical answers. Without it, Qwen samples randomly and your score would wobble between runs. `max_tokens=32` is plenty for `ANSWER: B` and stops a rambling model early.

Inspect writes a full **log** of every prompt, reply and score to `./logs/`. That log is the main product of an Inspect run.

In [ ]:
logs = eval(task, model=MODEL, model_args={"do_sample": False}, limit=LIMIT, max_tokens=32)
log = logs[0]
print("status:", log.status)
for name, metric in log.results.scores[0].metrics.items():
    print(f"{name:10s} {metric.value:.3f}")

## 4. What the model actually saw

Never trust a score you can't trace back to a real prompt and reply. Here is the exact conversation for the first question. The prompt text comes from the solver, not from you.

In [ ]:
first = log.samples[0]
for m in first.messages:
    print(f"--- {m.role} ---\n{m.text}\n")
score = next(iter(first.scores.values()))
print("scored:", score.value, "| parsed answer:", score.answer, "| target:", first.target)

## 5. Diagnostics

Inspect gives you accuracy and a standard error. The rest you get by reading the log. We turn each sample into a table row:

In [ ]:
import numpy as np, pandas as pd

LET = "ABCDEFGHIJKLM"
rows = []
for smp in log.samples:
    sc = next(iter(smp.scores.values()))
    target = smp.target[0] if isinstance(smp.target, list) else smp.target
    rows.append({"question": smp.input if isinstance(smp.input, str) else smp.input[-1].text,
                 "n_choices": len(smp.choices),
                 "target": target,
                 "answer": (sc.answer or "").strip() or None,   # None = no letter could be parsed
                 "correct": sc.value == "C",
                 "reply": smp.output.completion,
                 "choices": smp.choices})
df = pd.DataFrame(rows)
df[["question", "target", "answer", "correct"]].head()

### 5a. Accuracy, with an error bar and the chance baseline

TruthfulQA questions have different numbers of options (2 to 13), so **chance** is the average of 1/(options), not 25%. The 95% interval comes from a bootstrap: resample the questions 2,000 times and take the middle 95% of the scores.

In [ ]:
rng = np.random.default_rng(0)
c = df["correct"].to_numpy()
boot = rng.choice(c, size=(2000, len(c)), replace=True).mean(axis=1)
acc, lo, hi = c.mean(), *np.percentile(boot, [2.5, 97.5])
chance = (1 / df["n_choices"]).mean()
print(f"accuracy {acc:.3f}   95% CI [{lo:.3f}, {hi:.3f}]   n = {len(c)}")
print(f"chance   {chance:.3f}")

### 5b. Format compliance

Inspect's scorer only gives credit if it can find `ANSWER: <letter>` in the reply. A small model sometimes writes something else, and that counts as **wrong** even if it "knew" the answer. Separate the two before blaming the model's knowledge.

In [ ]:
unparsed = df[df["answer"].isna()]
print(f"format failures: {len(unparsed)} / {len(df)}  ({len(unparsed) / len(df):.1%})")
for r in unparsed["reply"].head(3):
    print("  reply:", repr(r[:120]))

### 5c. Position bias

Does the model favour a letter? Compare how often it **answers** each letter with how often each letter is **correct**. The shuffle spreads correct answers out, so a big spike in one predicted letter is the model, not the data.

In [ ]:
bias = pd.DataFrame({"answered": df["answer"].value_counts(normalize=True),
                     "correct": df["target"].value_counts(normalize=True)}).fillna(0).sort_index()
print(bias.head(6).round(3))
bias.head(6).plot.bar(rot=0, figsize=(6, 3.2), title="Answered vs correct letter")

### 5d. What it gets wrong

TruthfulQA's wrong options are mostly **popular misconceptions**. Every miss is a false belief the model picked over the true one. Read five. Is there a pattern: health myths, superstitions, misquotes, fiction taken literally?

In [ ]:
for _, r in df[~df["correct"] & df["answer"].notna()].head(5).iterrows():
    pick = r["choices"][LET.index(r["answer"][0])] if r["answer"][0] in LET[:r["n_choices"]] else "?"
    print("Q:", r["question"])
    print("   model chose:", pick)
    print("   true answer:", r["choices"][LET.index(r["target"])])
    print()

## 6. Same model, a second benchmark

Run ARC-Challenge (grade-school science) through the same pipeline, with the same solver and scorer and a different dataset. In Inspect, switching benchmarks is a one-line change. That's the point of using a framework.

**Is the model weak at everything, or specifically at not repeating misconceptions?** Compare each accuracy against its own chance level.

*If you also ran the optional "under the hood" notebook:* it scores ARC by reading the model's probabilities directly instead of parsing its reply. The two ARC numbers will differ. Same model and same questions, but a different measurement.

In [ ]:
from inspect_evals.arc import arc_challenge

arc_log = eval(arc_challenge(), model=MODEL, model_args={"do_sample": False}, limit=LIMIT, max_tokens=32)[0]
arc = {k: m.value for k, m in arc_log.results.scores[0].metrics.items()}
print(f"TruthfulQA MC1  accuracy {acc:.3f}   (chance {chance:.3f})")
print(f"ARC-Challenge   accuracy {arc['accuracy']:.3f}   (chance ~0.25)")

## 7. Save and bring Thursday

Saves a summary and downloads it, plus the Inspect log files. To browse a log visually, install the **Inspect AI** extension in VS Code and open the `.eval` file, or run `inspect view` on your laptop in the folder holding it.

In [ ]:
import json, glob, os
summary = {"model": MODEL, "framework": "Inspect + inspect_evals",
           "truthfulqa_mc1": {"n": int(len(df)), "accuracy": round(float(acc), 4),
                              "ci95": [round(float(lo), 4), round(float(hi), 4)],
                              "chance": round(float(chance), 4),
                              "format_failure_rate": round(len(unparsed) / len(df), 4),
                              "answered_share": bias["answered"].round(3).to_dict()},
           "arc_challenge": {k: round(float(v), 4) for k, v in arc.items()},
           "settings": {"decoding": "greedy (do_sample=False)", "max_tokens": 32, "limit": LIMIT}}
with open("summary_inspect.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

try:
    from google.colab import files
    files.download("summary_inspect.json")
    for p in sorted(glob.glob("logs/*.eval"), key=os.path.getmtime)[-2:]:
        files.download(p)
except ImportError:
    print("(not on Colab: files are in this folder and ./logs)")

## Optional: a bigger model

Change `MODEL` to `"hf/Qwen/Qwen2.5-1.5B-Instruct"` and run everything again. Does a 3× larger model believe fewer misconceptions? That's a two-point preview of the capability-gap question the AI Control project is built on.